# Hybrid LSTM-CNN for IoT Intrusion Detection
**Replicating Sinha et al., 2025** — *A High Performance Hybrid LSTM-CNN Secure Architecture for IoT Environments Using Deep Learning*

### All patches applied (v3 — full audit)
| # | Area | Change |
|---|------|--------|
| 1 | Kaggle auth | `KAGGLE_USERNAME` + `KAGGLE_KEY` |
| 2 | LabelEncoder import | Moved to `sklearn.preprocessing` |
| 3 | Normalization | `MinMaxScaler[0,1]` per Algorithm 2 |
| 4 | PCA leakage | Fit on X_train only |
| 5 | MIN_COMPONENTS | IndentationError fixed; guard block correct |
| 6 | MaxPool crash | `padding='same'` on all MaxPooling1D |
| 7 | Mixed precision | `dtype='float32'` on all output Dense layers |
| 8 | Attention layer | Added after LSTM block in LSTM-CNN (paper §3) |
| 9 | CNN LR | 0.001 → 0.0005 per Table 5 |
| 10 | Gradient clipping | `clipnorm=1.0` on ALL recurrent models |
| 11 | LR scheduler | ReduceLROnPlateau → CosineDecay per §4 |
| 12 | SHAP | Replaced permutation importance with SHAP KernelExplainer |
| 13 | Speed patches kept | EPOCHS=30, BATCH_SIZE=512, SAMPLE_FRAC=0.20, mixed_float16 |

---
Hit **Runtime → Run All** to execute.


In [ ]:
# CELL 1: Install Dependencies
%pip install -q numpy pandas scikit-learn scipy tensorflow keras matplotlib seaborn imbalanced-learn pyyaml kaggle joblib tqdm shap


In [ ]:
# CELL 2: Speed Patch + GPU Setup
# ── Mixed precision (biggest single speedup on T4) ───────────────────────────
import tensorflow as tf

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Compute dtype:  {tf.keras.mixed_precision.global_policy().compute_dtype}')
print(f'Variable dtype: {tf.keras.mixed_precision.global_policy().variable_dtype}')

# ── Verify GPU ────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('WARNING: No GPU detected — go to Runtime > Change runtime type > T4 GPU')


In [ ]:
# CELL 3: Kaggle Credentials and Dataset Download
import os, glob

os.environ['KAGGLE_USERNAME'] = 'kruinfosec'   # <-- replace
os.environ['KAGGLE_KEY']      = '0595034e9978007f27bea6c159f47f24'    # <-- replace

DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
    print('Downloading BoT-IoT dataset from Kaggle...')
    !kaggle datasets download -d majedjaber/bot-iot-all-features-5-sample -p {DATA_DIR} --unzip
else:
    n = len(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
    print(f'Dataset already present ({n} CSV files)')


In [ ]:
# CELL 4: Load Data
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
print(f'Loading {len(csv_files)} CSV file(s)...')
df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
print(f'Total records: {len(df):,}  |  Columns: {len(df.columns)}')

# Speed patch: 0.20 for Colab; set to 1.0 for full paper replication
SAMPLE_FRAC = 0.20
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    print(f'Sampled to {len(df):,} records ({SAMPLE_FRAC*100:.0f}%)')


In [ ]:
# CELL 5: Clean and Encode
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            df[col] = df[col].fillna(df[col].mean())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
print(f'After cleaning: {len(df):,} records')

# Auto-detect target column
target_col = None
for candidate in ['attack', 'Attack', 'label', 'Label', 'category', 'Category']:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    print('WARNING: could not auto-detect target. Available columns:')
    print(df.columns.tolist())
    target_col = input('Enter target column name: ')
print(f'Target: {target_col}\n{df[target_col].value_counts()}')

exclude_cols = {target_col}
for col in df.columns:
    if col.lower() in ['pkseqid','saddr','daddr','sport','dport','category','subcategory'] and col != target_col:
        exclude_cols.add(col)

cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns if c not in exclude_cols]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    print(f'One-hot encoded {len(cat_cols)} cols. Total features: {len(df.columns)}')

feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].values.astype(np.float32)
le = LabelEncoder()
y  = le.fit_transform(df[target_col])
print(f'Classes: {dict(zip(le.classes_, range(len(le.classes_))))}  |  X shape: {X.shape}')

N_CLASSES = len(np.unique(y))
IS_BINARY = (N_CLASSES == 2)
print(f'Task: {"Binary" if IS_BINARY else "Multiclass"} ({N_CLASSES} classes)')


In [ ]:
# CELL 6: Split, Scale, PCA, SMOTE
# FIX: MinMaxScaler [0,1] per paper Algorithm 2
# FIX: PCA fit only on X_train to prevent data leakage
# FIX: MIN_COMPONENTS guard — indentation corrected

# 70/15/15 split matching paper
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test,  y_val,  y_test  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Split: Train={len(X_train):,} | Val={len(X_val):,} | Test={len(X_test):,}')

# Min-Max scaling per paper Algorithm 2
scaler  = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)   # fit on train only — no leakage
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# PCA fit on train only — no leakage
# Save scaled arrays so the MIN_COMPONENTS guard can re-fit on original feature space
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

pca     = PCA(n_components=0.95, random_state=42)
X_train = pca.fit_transform(X_train_scaled)
X_val   = pca.transform(X_val_scaled)
X_test  = pca.transform(X_test_scaled)
print(f'PCA components retained: {X_train.shape[1]}')

# Guard: 3x MaxPool(2) requires sequence length >= 8 (2^3)
# FIX: Re-fit PCA on the scaled (pre-PCA) data so n_components <= n_features is guaranteed
MIN_COMPONENTS = 8
if X_train.shape[1] < MIN_COMPONENTS:
    n_comp = min(MIN_COMPONENTS, X_train_scaled.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    X_train = pca.fit_transform(X_train_scaled)
    X_val   = pca.transform(X_val_scaled)
    X_test  = pca.transform(X_test_scaled)
    print(f'PCA overridden to {n_comp} components to prevent MaxPool crash')

# SMOTE + random undersampling for class balance
print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
smote_pipeline = ImbPipeline([
    ('smote',       SMOTE(random_state=42)),
    ('undersample', RandomUnderSampler(random_state=42))
])
X_train, y_train = smote_pipeline.fit_resample(X_train, y_train)
print(f'After SMOTE:  {dict(pd.Series(y_train).value_counts())}')

# Reshape to 3-D for DL models: (samples, timesteps, features)
X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val_dl   = X_val.reshape((X_val.shape[0],     X_val.shape[1],   1))
X_test_dl  = X_test.reshape((X_test.shape[0],   X_test.shape[1],  1))
print(f'DL input shape: {X_train_dl.shape}')


In [ ]:
# CELL 7: Model Builders — all 6 architectures
# PATCH 1: Attention layer added to LSTM-CNN (paper §3 — key contribution)
# PATCH 2: CNN LR corrected to 0.0005 (paper Table 5)
# PATCH 3: clipnorm=1.0 added to ALL recurrent models (paper §4)
# PATCH 4: MaxPooling1D padding='same' on CNN + LSTM-CNN (prevents negative-dim crash)
# PATCH 5: dtype='float32' on all output Dense layers (required for mixed precision)

from tensorflow.keras import layers, models, regularizers

INPUT_SHAPE = (X_train_dl.shape[1], X_train_dl.shape[2])
N_OUT   = 1 if IS_BINARY else N_CLASSES
LOSS    = 'binary_crossentropy' if IS_BINARY else 'sparse_categorical_crossentropy'
OUT_ACT = 'sigmoid' if IS_BINARY else 'softmax'

# ── Cosine decay schedule ─────────────────────────────────────────────────────
# Paper §4: "initial learning rate set to 0.0005 and then reduced by cosine annealing"
# Steps computed at build time from training set size + batch size
STEPS_PER_EPOCH = max(1, len(X_train_dl) // 512)
TOTAL_STEPS     = 30 * STEPS_PER_EPOCH  # matches EPOCHS=30 speed patch

def cosine_decay(lr=0.0005):
    return tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr,
        decay_steps=TOTAL_STEPS,
        alpha=1e-6   # minimum LR floor
    )

# ── 1. LSTM-CNN Hybrid (proposed model) ──────────────────────────────────────
def build_lstm_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = inp

    # LSTM block — temporal feature extraction
    x = layers.LSTM(256, activation='tanh', return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_1')(x)
    x = layers.Dropout(0.3, name='lstm_drop_1')(x)
    x = layers.LSTM(128, activation='tanh', return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_2')(x)
    x = layers.Dropout(0.3, name='lstm_drop_2')(x)

    # PATCH 1: Attention mechanism (paper §3 — "attention structure to select
    # the most important features for classification purposes")
    x = layers.MultiHeadAttention(num_heads=4, key_dim=32, name='attention')(x, x)
    x = layers.LayerNormalization(name='attn_norm')(x)

    # Collapse sequence for CNN input
    x = layers.Lambda(lambda t: t[:, -1, :], name='seq_last')(x)   # take last timestep
    x = layers.Reshape((128, 1), name='reshape_for_cnn')(x)

    # CNN block — spatial feature extraction
    for filters, tag in zip([64, 128, 256], ['1', '2', '3']):
        x = layers.Conv1D(filters, 3, activation='relu', padding='same', name=f'conv_{tag}')(x)
        # PATCH 4: padding='same' prevents negative-dim crash when sequence < pool_size
        x = layers.MaxPooling1D(pool_size=2, padding='same', name=f'pool_{tag}')(x)

    x   = layers.Flatten(name='flatten')(x)
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4), name='dense_1')(x)
    x   = layers.Dropout(0.3, name='dense_drop')(x)
    # PATCH 5: dtype='float32' required for numerical stability with mixed precision
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32', name='output')(x)

    m = models.Model(inp, out, name='LSTM_CNN_Hybrid')
    # PATCH 3: clipnorm=1.0; PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.0005), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

# ── 2. Basic CNN ──────────────────────────────────────────────────────────────
def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = inp
    for f in [64, 128, 256]:
        x = layers.Conv1D(f, 3, activation='relu', padding='same')(x)
        # PATCH 4: padding='same'
        x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='CNN')
    # PATCH 2: LR 0.0005 per Table 5 (was 0.001)
    # PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.0005)),
              loss=LOSS, metrics=['accuracy'])
    return m

# ── 3. Simple RNN ─────────────────────────────────────────────────────────────
def build_rnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.SimpleRNN(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.SimpleRNN(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='RNN')
    # PATCH 3: clipnorm=1.0; PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.001), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

# ── 4. Standard LSTM ──────────────────────────────────────────────────────────
def build_lstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.LSTM(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='LSTM')
    # PATCH 3: clipnorm=1.0; PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.0005), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

# ── 5. Bidirectional LSTM ─────────────────────────────────────────────────────
def build_bilstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(128))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='BiLSTM')
    # PATCH 3: clipnorm=1.0; PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.0005), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

# ── 6. GRU ────────────────────────────────────────────────────────────────────
def build_gru(input_shape):
    inp = layers.Input(shape=input_shape)
    x   = layers.GRU(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.GRU(128)(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='GRU')
    # PATCH 3: clipnorm=1.0; PATCH scheduler: CosineDecay
    m.compile(optimizer=tf.keras.optimizers.Adam(cosine_decay(0.0005), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

ALL_MODELS = {
    'LSTM-CNN (Hybrid)': build_lstm_cnn,
    'CNN':               build_cnn,
    'RNN':               build_rnn,
    'LSTM':              build_lstm,
    'BiLSTM':            build_bilstm,
    'GRU':               build_gru,
}
print(f'Mixed precision policy: {tf.keras.mixed_precision.global_policy().name}')
print(f'Models defined: {list(ALL_MODELS.keys())}')


In [ ]:
# CELL 8: Training Loop
# Speed patches: EPOCHS=30 (paper=50), BATCH_SIZE=512 (paper=128)
# tf.data prefetch pipeline keeps GPU fed between batches

import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

EPOCHS     = 30   # paper uses 50; early stopping typically fires at 10–20
BATCH_SIZE = 512  # paper uses 128; larger batch fills T4 VRAM more efficiently

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, len(X)), seed=42)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_dl, y_train, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(X_val_dl,   y_val,   BATCH_SIZE)
test_ds  = make_dataset(X_test_dl,  y_test,  BATCH_SIZE)

all_results     = {}
all_histories   = {}
all_predictions = {}
trained_models  = {}

session_start = time.time()

for name, builder in ALL_MODELS.items():
    print(f'\n{"="*60}\n  Training: {name}\n{"="*60}')
    elapsed = (time.time() - session_start) / 60
    print(f'  Session time so far: {elapsed:.1f} min')

    model = builder(INPUT_SHAPE)
    model.summary()

    # Paper §4: early stopping on validation loss
    cbs = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5,
            restore_best_weights=True, verbose=1),
    ]
    # NOTE: ReduceLROnPlateau removed — LR schedule is now CosineDecay
    # (paper §4: "cosine annealing to speed up the convergence")

    start   = time.time()
    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=cbs,
        verbose=1
    )
    train_time = time.time() - start

    # ── Evaluate on test set ─────────────────────────────────────────────────
    y_proba = model.predict(test_ds, verbose=0)

    if IS_BINARY:
        y_proba_flat = y_proba.flatten()
        y_pred       = (y_proba_flat > 0.5).astype(int)
        auc_score    = roc_auc_score(y_test, y_proba_flat)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    else:
        y_pred       = np.argmax(y_proba, axis=1)
        y_proba_flat = y_proba.max(axis=1)
        try:
            auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        except Exception:
            auc_score = float('nan')
        cm_full = confusion_matrix(y_test, y_pred)
        fp = (cm_full.sum(axis=0) - np.diag(cm_full)).sum()
        fn = (cm_full.sum(axis=1) - np.diag(cm_full)).sum()
        tp = np.diag(cm_full).sum()
        tn = cm_full.sum() - fp - fn - tp

    avg     = 'binary' if IS_BINARY else 'macro'
    fpr_val = round((fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0, 2)
    dr_val  = round((tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0, 2)

    metrics = {
        'Accuracy (%)':       round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)':      round(precision_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'Recall (%)':         round(recall_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'F1-Score (%)':       round(f1_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'FPR (%)':            fpr_val,
        'Detection Rate (%)': dr_val,
        'AUC-ROC (%)':        round(auc_score * 100, 2),
        'Train Time (s)':     round(train_time, 1),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'Epochs Run': len(history.history['loss']),
    }

    all_results[name]     = metrics
    all_histories[name]   = history.history
    all_predictions[name] = {'y_true': y_test, 'y_pred': y_pred, 'y_proba': y_proba_flat}
    trained_models[name]  = model

    # Save model mid-loop — prevents data loss on Colab timeout
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    os.makedirs('models_checkpoint', exist_ok=True)
    model.save(f'models_checkpoint/{safe}.keras')

    print(f'\n  Accuracy:       {metrics["Accuracy (%)"]:.2f}%')
    print(f'  Precision:      {metrics["Precision (%)"]:.2f}%')
    print(f'  Recall:         {metrics["Recall (%)"]:.2f}%')
    print(f'  F1-Score:       {metrics["F1-Score (%)"]:.2f}%')
    print(f'  FPR:            {metrics["FPR (%)"]:.2f}%')
    print(f'  Detection Rate: {metrics["Detection Rate (%)"]:.2f}%')
    print(f'  AUC-ROC:        {metrics["AUC-ROC (%)"]:.2f}%')
    print(f'  Epochs run: {metrics["Epochs Run"]}  |  Time: {train_time:.0f}s')
    total_elapsed = (time.time() - session_start) / 60
    print(f'  Total session time: {total_elapsed:.1f} min')

print('\n' + '='*60 + '\n  ALL MODELS TRAINED!\n' + '='*60)
total_min = (time.time() - session_start) / 60
print(f'Total wall time: {total_min:.1f} minutes')


In [ ]:
# CELL 9: Results Comparison Table (matches Table 6 in paper)
display_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
                'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run']
results_df = pd.DataFrame(all_results).T[display_cols]
results_df.index.name = 'Model'

styled = (results_df.style
          .format('{:.2f}')
          .highlight_max(axis=0,
              subset=['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)','AUC-ROC (%)'],
              color='#c6efce')
          .highlight_min(axis=0, subset=['FPR (%)'], color='#c6efce')
          .set_caption('Performance Comparison — Sinha et al. (2025) Replication'))
display(styled)

# ── Target results from paper Table 6 for comparison ─────────────────────────
paper_targets = {
    'LSTM-CNN (Hybrid)': {'Accuracy (%)': 99.87, 'Precision (%)': 99.89, 'Recall (%)': 99.85, 'F1-Score (%)': 99.87, 'FPR (%)': 0.13},
    'BiLSTM':            {'Accuracy (%)': 98.92, 'Precision (%)': 98.97, 'Recall (%)': 98.87, 'F1-Score (%)': 98.92, 'FPR (%)': 1.03},
    'LSTM':              {'Accuracy (%)': 98.34, 'Precision (%)': 98.42, 'Recall (%)': 98.26, 'F1-Score (%)': 98.34, 'FPR (%)': 1.58},
    'GRU':               {'Accuracy (%)': 97.89, 'Precision (%)': 97.95, 'Recall (%)': 97.83, 'F1-Score (%)': 97.89, 'FPR (%)': 2.05},
    'CNN':               {'Accuracy (%)': 97.45, 'Precision (%)': 97.62, 'Recall (%)': 97.28, 'F1-Score (%)': 97.45, 'FPR (%)': 2.38},
    'RNN':               {'Accuracy (%)': 95.78, 'Precision (%)': 95.91, 'Recall (%)': 95.65, 'F1-Score (%)': 95.78, 'FPR (%)': 4.09},
}
paper_df = pd.DataFrame(paper_targets).T[['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'FPR (%)']]
paper_df.index.name = 'Model'
print('\n--- Paper target (Table 6) ---')
display(paper_df.style.format('{:.2f}').set_caption('Paper Table 6 — Target'))


In [ ]:
# CELL 10: Visualizations (matches Figures 6–8 in paper)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

# ── 10a. LSTM-CNN training history ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = all_histories['LSTM-CNN (Hybrid)']
ax1.plot(h['loss'],         label='Train', color=COLORS[0], lw=2)
ax1.plot(h['val_loss'],     label='Val',   color=COLORS[3], lw=2)
ax1.set_title('LSTM-CNN Hybrid — Loss',     fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.plot(h['accuracy'],     label='Train', color=COLORS[1], lw=2)
ax2.plot(h['val_accuracy'], label='Val',   color=COLORS[4], lw=2)
ax2.set_title('LSTM-CNN Hybrid — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

# ── 10b. ROC Curves ───────────────────────────────────────────────────────────
if IS_BINARY:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (name, data) in enumerate(all_predictions.items()):
        fpr_v, tpr_v, _ = roc_curve(data['y_true'], data['y_proba'])
        ax.plot(fpr_v, tpr_v, color=COLORS[i%6], lw=2,
                label=f'{name} (AUC={auc(fpr_v,tpr_v):.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — All Models', fontweight='bold')
    ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

# ── 10c. Bar chart comparison ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_cols = ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)']
results_df[bar_cols].plot(kind='bar', ax=ax, color=COLORS[:5], width=0.75)
ax.set_ylabel('Score (%)'); ax.set_title('Model Comparison — All Metrics', fontweight='bold')
ax.set_ylim([max(results_df[bar_cols].min().min()-2, 80), 101])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=6, padding=2)
ax.legend(fontsize=9, loc='lower right'); plt.tight_layout(); plt.show()

# ── 10d. Confusion matrices ───────────────────────────────────────────────────
n_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, (name, data) in enumerate(all_predictions.items()):
    ax = axes[idx]
    cm = confusion_matrix(data['y_true'], data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    m = all_results[name]
    ax.set_title(f'{name}\nAcc={m["Accuracy (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%  FPR={m["FPR (%)"]:.2f}%',
                 fontweight='bold', fontsize=10)
for idx in range(n_models, len(axes)): axes[idx].set_visible(False)
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── 10e. Classification reports ───────────────────────────────────────────────
print('\n' + '='*60 + '\n  CLASSIFICATION REPORTS\n' + '='*60)
for name, data in all_predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(data['y_true'], data['y_pred'],
          target_names=[str(c) for c in le.classes_], digits=4))


In [ ]:
# CELL 11: Learning Curves Comparison (Figure 8 in paper)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
LINE_STYLES = ['-','--','-.', ':', '-','--']
MARKERS     = ['o','s','^', 'D','v', 'P']
for (metric_key, title, ylabel, ax) in [
    ('loss',         'Training Loss',       'Loss',     axes[0,0]),
    ('val_loss',     'Validation Loss',     'Loss',     axes[0,1]),
    ('accuracy',     'Training Accuracy',   'Accuracy', axes[1,0]),
    ('val_accuracy', 'Validation Accuracy', 'Accuracy', axes[1,1]),
]:
    for i, (name, hist) in enumerate(all_histories.items()):
        ep = range(1, len(hist[metric_key])+1)
        ax.plot(ep, hist[metric_key], color=COLORS[i%6], lw=2,
                linestyle=LINE_STYLES[i%6], marker=MARKERS[i%6],
                markersize=4, markevery=max(1, len(hist[metric_key])//10),
                label=name, alpha=0.9)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# CELL 12: SHAP Feature Importance (Figure 13 in paper)
# PATCH: Replaced permutation importance with SHAP KernelExplainer
# Paper: "SHAP-based feature importance analysis revealing packet size,
#         connection duration, and protocol type as key indicators"

import shap
import warnings
warnings.filterwarnings('ignore')

# Use a small background sample for KernelExplainer (speed vs accuracy trade-off)
N_BACKGROUND = min(100, len(X_test))
N_EXPLAIN    = min(200, len(X_test))

background = X_test[:N_BACKGROUND]  # 2-D array (no reshape needed for wrapper)
explain_X  = X_test[:N_EXPLAIN]

# Wrapper: SHAP needs a 2-D predict function; model expects 3-D input
def predict_fn(X_2d):
    X_3d = X_2d.reshape(X_2d.shape[0], X_2d.shape[1], 1)
    proba = trained_models['LSTM-CNN (Hybrid)'].predict(X_3d, verbose=0)
    if IS_BINARY:
        return proba.flatten()
    return proba

print('Computing SHAP values (KernelExplainer) — this may take a few minutes...')
explainer   = shap.KernelExplainer(predict_fn, background)
shap_values = explainer.shap_values(explain_X, nsamples=100, l1_reg='aic')

# Map PCA component indices back to readable names
pca_feature_names = [f'PC_{i+1}' for i in range(X_test.shape[1])]

# ── SHAP bar plot — mean absolute SHAP values per feature ─────────────────────
if isinstance(shap_values, list):
    sv = np.abs(shap_values[1]).mean(axis=0)   # class-1 for binary
else:
    sv = np.abs(shap_values).mean(axis=0)

sorted_idx   = np.argsort(sv)[::-1][:20]
top_features = [pca_feature_names[i] for i in sorted_idx]
top_shap     = sv[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_shap, color='#2196F3', alpha=0.85)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('LSTM-CNN Hybrid — Top 20 Feature Importance (SHAP)', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 10 features by SHAP importance:')
for i in range(min(10, len(top_features))):
    print(f'  {i+1:2d}. {top_features[i]}: {top_shap[i]:.4f}')


In [ ]:
# CELL 13: Save All Results
import json
from datetime import datetime

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR     = f'results/run_{timestamp}'
for d in [f'{RUN_DIR}/figures', f'{RUN_DIR}/tables', f'{RUN_DIR}/models', f'{RUN_DIR}/training_history']:
    os.makedirs(d, exist_ok=True)
print(f'Saving to: {RUN_DIR}/')

# CSV + LaTeX tables
results_df.to_csv(f'{RUN_DIR}/tables/model_comparison.csv')
results_df.to_latex(f'{RUN_DIR}/tables/model_comparison.tex', float_format='%.2f')

# Full results JSON
run_meta = dict(
    timestamp=timestamp, sample_frac=SAMPLE_FRAC,
    scaler='MinMaxScaler[0,1]', scheduler='CosineDecay',
    epochs_max=EPOCHS, batch_size=BATCH_SIZE,
    task='binary' if IS_BINARY else 'multiclass',
    patches_applied=[
        'attention_layer_in_lstm_cnn',
        'cnn_lr_0.0005',
        'clipnorm_all_recurrent',
        'cosine_decay_scheduler',
        'shap_feature_importance',
        'maxpool_padding_same',
        'min_components_guard',
        'mixed_float16',
    ]
)
full_results = {
    'run_config': run_meta,
    'metrics': {
        n: {k: float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v
            for k, v in m.items()}
        for n, m in all_results.items()
    }
}
with open(f'{RUN_DIR}/tables/all_results.json', 'w') as f:
    json.dump(full_results, f, indent=2)

# Training histories
for name, hist in all_histories.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    with open(f'{RUN_DIR}/training_history/{safe}_history.json', 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in hist.items()}, f, indent=2)

# Classification reports
with open(f'{RUN_DIR}/tables/classification_reports.txt', 'w') as f:
    for name, data in all_predictions.items():
        f.write(f'\n{"="*60}\n  {name}\n{"="*60}\n')
        f.write(classification_report(data['y_true'], data['y_pred'],
                target_names=[str(c) for c in le.classes_], digits=4))

# Trained models
for name, model in trained_models.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    model.save(f'{RUN_DIR}/models/{safe}.keras')

print('All results saved.')
for root, dirs, files in os.walk(RUN_DIR):
    lvl = root.replace(RUN_DIR, '').count(os.sep)
    print('  '*lvl + os.path.basename(root) + '/')
    for f in sorted(files):
        kb = os.path.getsize(os.path.join(root, f)) / 1024
        print('  '*(lvl+1) + f + f' ({kb:.1f} KB)')


## Target Results (paper Table 6)

| Model | Accuracy | Precision | Recall | F1 | FPR |
|-------|----------|-----------|--------|----|-----|
| **LSTM-CNN (Hybrid)** | **99.87%** | **99.89%** | **99.85%** | **99.87%** | **0.13%** |
| BiLSTM | 98.92% | 98.97% | 98.87% | 98.92% | 1.03% |
| LSTM | 98.34% | 98.42% | 98.26% | 98.34% | 1.58% |
| GRU | 97.89% | 97.95% | 97.83% | 97.89% | 2.05% |
| CNN | 97.45% | 97.62% | 97.28% | 97.45% | 2.38% |
| RNN | 95.78% | 95.91% | 95.65% | 95.78% | 4.09% |

**Note**: Speed patch uses SAMPLE_FRAC=0.20 — set to 1.0 for full dataset replication.  
**Next step**: Quantum encoder integration — attach to the LSTM output (post-attention, pre-CNN) for hybrid quantum-classical forward pass.
